# SimCBAM study — corrected pipeline (v3: stable training protocol)

**Why this notebook exists.** The original pipeline saved the sparse FBP inputs with `sparse.astype(np.uint8)`. FBP values outside 0–255 (negative ringing, overshoot) *wrap around* (e.g. -3 → 253), which puts isolated saturated white pixels into the network inputs (about 10% of pixels at 32 angles). This notebook keeps the inputs as **float32**, so nothing is wrapped or rounded.

**What changes versus the original notebook**
1. Sparse inputs stored as float32 `.npy` (no uint8 cast).
2. `CIRCLE = False`: with `circle=True` anything outside the inscribed circle (~21% of the image) is *lost* from the input, yet the ground truth still contains vessels there, so the networks had to invent them. Set `CIRCLE = True` only if you deliberately want the old behaviour.
3. Random seeds, resume-safe training, per-image PSNR/SSIM saved (for paired statistics).
4. Metrics computed on outputs clipped to [0, 1].
5. Figures regenerated at printed size with correct labels (projection angles, not "sensor angles"; axes not swapped; fixed grey scale for all panels).

**v3 additions (after the first corrected run).** The first run showed unstable final epochs (validation loss jumping 2–25× between epochs), so the reported *last-epoch* number was largely noise: e.g. SimCBAM-96 had 37.0 dB validation PSNR at epoch 20 but 33.0 dB at epoch 30. v3 therefore (a) **keeps the checkpoint with the lowest validation loss**, (b) uses **cosine learning-rate decay**, (c) runs **3 seeds** by default, and (d) stores the old-style last-epoch numbers too, for transparency. The data from the first run are reused (no regeneration).

**Model code is copied verbatim from the original notebook**, and the cell after it *asserts* the parameter counts (13,568 / 14,254 / 0 / 686).

Runtime on a Colab L4: roughly 1.5–2.5 h for one seed (15 models). Everything is saved to `OUT` after every model, so you can stop and resume.

In [ ]:
# ---------- 0. setup ----------
import os, json, time, random, math, numpy as np
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm import tqdm
from skimage.transform import radon, iradon
from skimage.metrics import peak_signal_noise_ratio as psnr_fn, structural_similarity as ssim_fn
import matplotlib as mpl
import matplotlib.pyplot as plt


device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
torch.backends.cudnn.benchmark = True
print('device:', device)

# ---- settings -------------------------------------------------------------
ROOT   = os.path.expanduser(os.environ.get('VESSEL_ROOT', '../data/Vessel'))   # folder with train/ test/ validation/ .tif (Tong et al. Vessel set)
OUT    = os.path.expanduser(os.environ.get('SIMCBAM_OUT', '../runs/v3'))      # .npy data, ckpt/, results/, figures/ are written here
ANGLES = [32, 64, 96]
CIRCLE = False      # see note above; False = no truncation outside the inscribed circle
SEEDS  = [0, 1, 2]  # mean +/- SD across training runs. Tip: set [0] first, check it works, then set [0, 1, 2] and re-run (finished runs are skipped)
PROTOCOL = 'v3'     # tag in file names, so the first run's files are never reused or overwritten
EPOCHS, BATCH, LR = 30, 16, 1e-3                       # same as the original experiments
for d in ['', 'ckpt', 'results', 'figures']:
    os.makedirs(f'{OUT}/{d}', exist_ok=True)

## 1. Model definitions (copied verbatim from the original notebook) + parameter-count check

In [ ]:
# --- U-Net architecture ---
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.conv(x)

class UNet(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, base=32):
        super().__init__()
        self.enc1 = DoubleConv(in_ch, base)
        self.enc2 = DoubleConv(base, base*2)
        self.enc3 = DoubleConv(base*2, base*4)
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = DoubleConv(base*4, base*8)
        self.up3 = nn.ConvTranspose2d(base*8, base*4, 2, stride=2)
        self.dec3 = DoubleConv(base*8, base*4)
        self.up2 = nn.ConvTranspose2d(base*4, base*2, 2, stride=2)
        self.dec2 = DoubleConv(base*4, base*2)
        self.up1 = nn.ConvTranspose2d(base*2, base, 2, stride=2)
        self.dec1 = DoubleConv(base*2, base)
        self.out_conv = nn.Conv2d(base, out_ch, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        b = self.bottleneck(self.pool(e3))
        d3 = self.up3(b)
        d3 = self.dec3(torch.cat([d3, e3], dim=1))
        d2 = self.up2(d3)
        d2 = self.dec2(torch.cat([d2, e2], dim=1))
        d1 = self.up1(d2)
        d1 = self.dec1(torch.cat([d1, e1], dim=1))
        return self.out_conv(d1)


class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.shape
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y.expand_as(x)

class DoubleConvSE(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )
        self.se = SEBlock(out_ch)

    def forward(self, x):
        x = self.conv(x)
        return self.se(x)

class UNet_SE(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, base=32):
        super().__init__()
        self.enc1 = DoubleConvSE(in_ch, base)
        self.enc2 = DoubleConvSE(base, base*2)
        self.enc3 = DoubleConvSE(base*2, base*4)
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = DoubleConvSE(base*4, base*8)
        self.up3 = nn.ConvTranspose2d(base*8, base*4, 2, stride=2)
        self.dec3 = DoubleConvSE(base*8, base*4)
        self.up2 = nn.ConvTranspose2d(base*4, base*2, 2, stride=2)
        self.dec2 = DoubleConvSE(base*4, base*2)
        self.up1 = nn.ConvTranspose2d(base*2, base, 2, stride=2)
        self.dec1 = DoubleConvSE(base*2, base)
        self.out_conv = nn.Conv2d(base, out_ch, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        b = self.bottleneck(self.pool(e3))
        d3 = self.up3(b)
        d3 = self.dec3(torch.cat([d3, e3], dim=1))
        d2 = self.up2(d3)
        d2 = self.dec2(torch.cat([d2, e2], dim=1))
        d1 = self.up1(d2)
        d1 = self.dec1(torch.cat([d1, e1], dim=1))
        return self.out_conv(d1)


class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False)
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        b, c, _, _ = x.shape
        avg_out = self.fc(self.avg_pool(x).view(b, c))
        max_out = self.fc(self.max_pool(x).view(b, c))
        out = self.sigmoid(avg_out + max_out).view(b, c, 1, 1)
        return x * out

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size//2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        out = torch.cat([avg_out, max_out], dim=1)
        out = self.sigmoid(self.conv(out))
        return x * out

class CBAM(nn.Module):
    def __init__(self, channels, reduction=16, kernel_size=7):
        super().__init__()
        self.ca = ChannelAttention(channels, reduction)
        self.sa = SpatialAttention(kernel_size)

    def forward(self, x):
        x = self.ca(x)
        x = self.sa(x)
        return x

class DoubleConvCBAM(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )
        self.cbam = CBAM(out_ch)

    def forward(self, x):
        x = self.conv(x)
        return self.cbam(x)

class UNet_CBAM(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, base=32):
        super().__init__()
        self.enc1 = DoubleConvCBAM(in_ch, base)
        self.enc2 = DoubleConvCBAM(base, base*2)
        self.enc3 = DoubleConvCBAM(base*2, base*4)
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = DoubleConvCBAM(base*4, base*8)
        self.up3 = nn.ConvTranspose2d(base*8, base*4, 2, stride=2)
        self.dec3 = DoubleConvCBAM(base*8, base*4)
        self.up2 = nn.ConvTranspose2d(base*4, base*2, 2, stride=2)
        self.dec2 = DoubleConvCBAM(base*4, base*2)
        self.up1 = nn.ConvTranspose2d(base*2, base, 2, stride=2)
        self.dec1 = DoubleConvCBAM(base*2, base)
        self.out_conv = nn.Conv2d(base, out_ch, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        b = self.bottleneck(self.pool(e3))
        d3 = self.up3(b)
        d3 = self.dec3(torch.cat([d3, e3], dim=1))
        d2 = self.up2(d3)
        d2 = self.dec2(torch.cat([d2, e2], dim=1))
        d1 = self.up1(d2)
        d1 = self.dec1(torch.cat([d1, e1], dim=1))
        return self.out_conv(d1)


class SimAM(nn.Module):
    def __init__(self, e_lambda=1e-4):
        super().__init__()
        self.e_lambda = e_lambda
        self.activation = nn.Sigmoid()

    def forward(self, x):
        b, c, h, w = x.shape
        n = w * h - 1
        x_minus_mu_square = (x - x.mean(dim=[2, 3], keepdim=True)).pow(2)
        y = x_minus_mu_square / (4 * (x_minus_mu_square.sum(dim=[2, 3], keepdim=True) / n + self.e_lambda)) + 0.5
        return x * self.activation(y)

class DoubleConvSimAM(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True)
        )
        self.simam = SimAM()
    def forward(self, x):
        return self.simam(self.conv(x))

class UNet_SimAM(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, base=32):
        super().__init__()
        self.enc1 = DoubleConvSimAM(in_ch, base)
        self.enc2 = DoubleConvSimAM(base, base*2)
        self.enc3 = DoubleConvSimAM(base*2, base*4)
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = DoubleConvSimAM(base*4, base*8)
        self.up3 = nn.ConvTranspose2d(base*8, base*4, 2, stride=2)
        self.dec3 = DoubleConvSimAM(base*8, base*4)
        self.up2 = nn.ConvTranspose2d(base*4, base*2, 2, stride=2)
        self.dec2 = DoubleConvSimAM(base*4, base*2)
        self.up1 = nn.ConvTranspose2d(base*2, base, 2, stride=2)
        self.dec1 = DoubleConvSimAM(base*2, base)
        self.out_conv = nn.Conv2d(base, out_ch, 1)
    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        b = self.bottleneck(self.pool(e3))
        d3 = self.dec3(torch.cat([self.up3(b), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.out_conv(d1)


class SimCBAM(nn.Module):
    """Hybrid: SimAM (parameter-free 'channel-like' attention) + CBAM's spatial attention branch"""
    def __init__(self, channels, kernel_size=7):
        super().__init__()
        self.simam = SimAM()
        self.sa = SpatialAttention(kernel_size)
    def forward(self, x):
        x = self.simam(x)
        x = self.sa(x)
        return x

class DoubleConvSimCBAM(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True)
        )
        self.simcbam = SimCBAM(out_ch)
    def forward(self, x):
        return self.simcbam(self.conv(x))

class UNet_SimCBAM(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, base=32):
        super().__init__()
        self.enc1, self.enc2, self.enc3 = DoubleConvSimCBAM(in_ch, base), DoubleConvSimCBAM(base, base*2), DoubleConvSimCBAM(base*2, base*4)
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = DoubleConvSimCBAM(base*4, base*8)
        self.up3, self.dec3 = nn.ConvTranspose2d(base*8, base*4, 2, stride=2), DoubleConvSimCBAM(base*8, base*4)
        self.up2, self.dec2 = nn.ConvTranspose2d(base*4, base*2, 2, stride=2), DoubleConvSimCBAM(base*4, base*2)
        self.up1, self.dec1 = nn.ConvTranspose2d(base*2, base, 2, stride=2), DoubleConvSimCBAM(base*2, base)
        self.out_conv = nn.Conv2d(base, out_ch, 1)
    def forward(self, x):
        e1 = self.enc1(x); e2 = self.enc2(self.pool(e1)); e3 = self.enc3(self.pool(e2))
        b = self.bottleneck(self.pool(e3))
        d3 = self.dec3(torch.cat([self.up3(b), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.out_conv(d1)

In [ ]:
MODELS = {'Baseline': UNet, 'SE': UNet_SE, 'CBAM': UNet_CBAM, 'SimAM': UNet_SimAM, 'SimCBAM': UNet_SimCBAM}
count = lambda m: sum(p.numel() for p in m.parameters() if p.requires_grad)
base = count(UNet())
rows = []
for k, c in MODELS.items():
    n = count(c())
    rows.append((k, n, n - base, 100 * (n - base) / base))
    print(f'{k:8s} total {n:>10,}   extra {n - base:>7,}   ({100 * (n - base) / base:5.2f}% of baseline)')
# the numbers quoted in the paper; the notebook stops here if they are not reproduced
assert [r[2] for r in rows] == [0, 13568, 14254, 0, 686], 'parameter counts differ from the manuscript!'
json.dump(rows, open(f'{OUT}/results/parameter_counts.json', 'w'))
print('OK: parameter counts reproduced.')

## 2. Data: sparse inputs as float32 (the fix)
Also prints a diagnostic that shows exactly what the old `astype(np.uint8)` did.

In [ ]:
def load_gray(path):
    return np.array(Image.open(path)).astype(np.float64) / 255.0

def make_sparse(gt, n_angles, circle=CIRCLE):
    theta = np.linspace(0., 180., n_angles, endpoint=False)
    sino = radon(gt, theta=theta, circle=circle)
    return iradon(sino, theta=theta, circle=circle).astype(np.float32)      # FLOAT: no uint8 cast, no rounding

def build_split(split):
    files = sorted(f for f in os.listdir(f'{ROOT}/{split}') if f.endswith('.tif'))
    gts = np.stack([load_gray(f'{ROOT}/{split}/{f}') for f in files])
    np.save(f'{OUT}/gt_{split}.npy', gts.astype(np.float32))
    for n in ANGLES:
        sp = np.stack([make_sparse(g, n) for g in tqdm(gts, desc=f'{split} {n} angles')])
        np.save(f'{OUT}/sparse{n}_{split}.npy', sp)

need_tifs = not os.path.exists(f'{OUT}/sparse96_test.npy')

if need_tifs:
    for split in ['train', 'validation', 'test']:
        build_split(split)
else:
    print('sparse data already generated (the original .tif files are not needed for training)')

# ---- diagnostic: what the OLD code did to the inputs ----
if os.path.exists(f'{ROOT}/test/0.tif'):
    g = load_gray(f'{ROOT}/test/0.tif')
    for n in ANGLES:
        th = np.linspace(0., 180., n, endpoint=False)
        rec = iradon(radon(g * 255, theta=th, circle=True), theta=th, circle=True)           # old scale 0..255
        bad = ((rec < 0) | (rec > 255)).mean() * 100
        print(f'old pipeline, {n:3d} angles: {bad:4.1f}% of pixels were outside 0..255 and were wrapped by astype(uint8)')
else:
    print('original .tif files not in this runtime: diagnostic skipped (optional; already shown in your first run)')

In [ ]:
class NPYDataset(Dataset):
    def __init__(self, n_angles, split):
        self.x = np.load(f'{OUT}/sparse{n_angles}_{split}.npy', mmap_mode='r')
        self.y = np.load(f'{OUT}/gt_{split}.npy', mmap_mode='r')
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        return torch.from_numpy(np.array(self.x[i]))[None], torch.from_numpy(np.array(self.y[i]))[None]

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

@torch.no_grad()
def evaluate(model, loader):
    model.eval(); P, S_ = [], []
    for x, y in loader:
        out = model(x.to(device)).clamp(0, 1).cpu().numpy(); y = y.numpy()
        for i in range(out.shape[0]):
            P.append(float(psnr_fn(y[i, 0], out[i, 0], data_range=1.0)))
            S_.append(float(ssim_fn(y[i, 0], out[i, 0], data_range=1.0)))
    return P, S_

def run_one(model_name, n_angles, seed):
    tag = f'{PROTOCOL}_{model_name}_{n_angles}_seed{seed}'
    rs, ck = f'{OUT}/results/{tag}.json', f'{OUT}/ckpt/{tag}.pth'
    if os.path.exists(rs):
        return json.load(open(rs))                                   # resume-safe
    set_seed(seed)
    g = torch.Generator().manual_seed(seed)
    tr = DataLoader(NPYDataset(n_angles, 'train'), batch_size=BATCH, shuffle=True, generator=g, num_workers=0)
    va = DataLoader(NPYDataset(n_angles, 'validation'), batch_size=BATCH)
    te = DataLoader(NPYDataset(n_angles, 'test'), batch_size=BATCH)
    model = MODELS[model_name](in_ch=1, out_ch=1, base=32).to(device)
    opt = optim.Adam(model.parameters(), lr=LR)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)       # smooth decay -> stable last epochs
    crit, t0 = nn.MSELoss(), time.time()
    best_val, best_ep, best_state, hist = float('inf'), 0, None, []
    for ep in range(EPOCHS):
        model.train(); tl = 0.0
        for x, y in tr:
            x, y = x.to(device), y.to(device)
            opt.zero_grad(); loss = crit(model(x), y); loss.backward(); opt.step()
            tl += loss.item() * x.size(0)
        sched.step()
        model.eval(); vl = 0.0
        with torch.no_grad():
            for x, y in va:
                vl += crit(model(x.to(device)), y.to(device)).item() * x.size(0)
        vl /= len(va.dataset); tl /= len(tr.dataset)
        hist.append(dict(epoch=ep + 1, train=tl, val=vl))
        if vl < best_val:                                              # keep the best-validation weights
            best_val, best_ep = vl, ep + 1
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        if (ep + 1) % 5 == 0 or ep == 0:
            print(f'[{tag}] epoch {ep + 1}/{EPOCHS}  train {tl:.5f}  val {vl:.5f}  (best val {best_val:.5f} @ epoch {best_ep})')
    P_last, S_last = evaluate(model, te)                              # what the first run reported (last epoch)
    model.load_state_dict(best_state); torch.save(best_state, ck)
    P, S_ = evaluate(model, te)                                       # reported number: best-validation checkpoint
    res = dict(model=model_name, angles=n_angles, seed=seed, params=count(model), train_seconds=time.time() - t0,
               best_epoch=best_ep, best_val=best_val,
               psnr_mean=float(np.mean(P)), ssim_mean=float(np.mean(S_)),
               psnr_last_epoch=float(np.mean(P_last)), ssim_last_epoch=float(np.mean(S_last)),
               history=hist, psnr_per_image=P, ssim_per_image=S_)
    json.dump(res, open(rs, 'w'))
    print(f'==> {tag}: PSNR {res["psnr_mean"]:.2f} dB | SSIM {res["ssim_mean"]:.4f}   (last epoch would have been {res["psnr_last_epoch"]:.2f} dB / {res["ssim_last_epoch"]:.4f}; best epoch {best_ep})\n')
    return res

## 3. Train and evaluate all models (resumable: re-run this cell if the runtime disconnects)

In [ ]:
for seed in SEEDS:
    for n in ANGLES:
        for name in MODELS:
            run_one(name, n, seed)
print('all done')

## 4. Results table (mean over test images; mean ± SD across seeds if several seeds were run)

In [ ]:
import glob, pandas as pd
R = [json.load(open(f)) for f in sorted(glob.glob(f'{OUT}/results/{PROTOCOL}_*_seed*.json'))]
df = pd.DataFrame([{k: v for k, v in r.items() if k not in ('psnr_per_image', 'ssim_per_image', 'history')} for r in R])
agg = df.groupby(['angles', 'model']).agg(PSNR=('psnr_mean', 'mean'), PSNR_sd=('psnr_mean', 'std'),
                                         SSIM=('ssim_mean', 'mean'), SSIM_sd=('ssim_mean', 'std'),
                                         PSNR_last_epoch=('psnr_last_epoch', 'mean'), best_epoch=('best_epoch', 'mean'),
                                         params=('params', 'first'), runs=('seed', 'count')).reset_index()
base_p = df[df.model == 'Baseline'].params.iloc[0]
agg['extra_params'] = agg.params - base_p
order = {m: i for i, m in enumerate(MODELS)}
agg = agg.sort_values(['angles', 'model'], key=lambda s: s.map(order) if s.name == 'model' else s)
agg.to_csv(f'{OUT}/results/summary_table.csv', index=False)
pd.set_option('display.width', 200)
print(agg.round({'PSNR': 2, 'PSNR_sd': 3, 'SSIM': 4, 'SSIM_sd': 4, 'PSNR_last_epoch': 2, 'best_epoch': 1}).to_string(index=False))

## 5. Statistics: paired comparison over the 200 test images
Per-image differences (same test image, two models). This does **not** replace repeated training runs (use `SEEDS = [0,1,2]` for that), but it gives a valid paired test and confidence interval for the image-level variation. Holm correction is applied within each metric. The Wilcoxon test is about the *median* difference while the bootstrap interval is for the *mean* difference; when they disagree the per-image differences are skewed. A tiny but consistent per-image difference can be 'significant' without being practically meaningful.

In [ ]:
from scipy.stats import wilcoxon
def per_image(model, n, key):
    return np.mean([json.load(open(f'{OUT}/results/{PROTOCOL}_{model}_{n}_seed{sd}.json'))[key] for sd in SEEDS], axis=0)

def holm(p):
    p = np.asarray(p); o = np.argsort(p); m = len(p); adj = np.empty(m); run = 0
    for rank, i in enumerate(o):
        run = max(run, (m - rank) * p[i]); adj[i] = min(1, run)
    return adj

rng = np.random.default_rng(0); rows = []
for n in ANGLES:
    for key, label in [('psnr_per_image', 'PSNR'), ('ssim_per_image', 'SSIM')]:
        for ref in ['Baseline', 'CBAM']:
            for m in MODELS:
                if m == ref: continue
                d = per_image(m, n, key) - per_image(ref, n, key)
                boots = [rng.choice(d, len(d)).mean() for _ in range(2000)]
                rows.append(dict(angles=n, metric=label, model=m, vs=ref, mean_diff=d.mean(), median_diff=np.median(d),
                                 ci_low=np.percentile(boots, 2.5), ci_high=np.percentile(boots, 97.5),
                                 p_raw=wilcoxon(d).pvalue))
T = pd.DataFrame(rows)
T['p_holm'] = T.groupby(['angles', 'metric', 'vs']).p_raw.transform(lambda s: holm(s.values))
T.to_csv(f'{OUT}/results/paired_tests.csv', index=False)
pd.set_option('display.float_format', lambda v: f'{v:.4g}')
print(T.to_string(index=False))

## 6. Computational cost that was actually measured (FLOPs, latency, GPU memory)
Needed if the paper says anything about efficiency. FLOPs count convolution / linear layers only (element-wise attention operations are not included by the counter).

In [ ]:
from torch.utils.flop_counter import FlopCounterMode
x = torch.randn(1, 1, 128, 128, device=device); cost = []
for name, cls in MODELS.items():
    m = cls().to(device).eval()
    with torch.no_grad(), FlopCounterMode(display=False) as fc:
        m(x)
    flops = fc.get_total_flops()
    if device.type == 'cuda': torch.cuda.reset_peak_memory_stats()
    with torch.no_grad():
        for _ in range(30): m(x)
        if device.type == 'cuda': torch.cuda.synchronize()
        t0 = time.time()
        for _ in range(200): m(x)
        if device.type == 'cuda': torch.cuda.synchronize()
    ms = (time.time() - t0) / 200 * 1000
    mem = torch.cuda.max_memory_allocated() / 2**20 if device.type == 'cuda' else float('nan')
    cost.append(dict(model=name, GFLOPs=flops / 1e9, latency_ms=ms, peak_mem_MB=mem))
cost = pd.DataFrame(cost); cost.to_csv(f'{OUT}/results/cost.csv', index=False); print(cost.round(3).to_string(index=False))

## 7. Figures (printed width 13.3 cm, readable fonts, corrected labels)
Changes versus the old figures: panels titled with the real content; Radon data axes are **projection angle (x)** and **detector position (y)**; every image uses the same fixed grey scale 0–1 (the old code auto-scaled each panel, which makes models look different for the wrong reason); nothing says "sensor", "acoustic" or "photoacoustic".

In [ ]:
mpl.rcParams.update({'font.size': 6.5, 'axes.titlesize': 6.8, 'axes.labelsize': 6.3, 'xtick.labelsize': 5.6,
                     'ytick.labelsize': 5.6, 'savefig.dpi': 300, 'font.family': 'DejaVu Sans'})
W_IN = 5.2          # 13.3 cm
IDX  = 0            # which test image to show (same one in every figure)
FIG  = f'{OUT}/figures'
gt_test = np.load(f'{OUT}/gt_test.npy')
G = gt_test[IDX].astype(np.float64)

def radon_pair(gt, n):
    th = np.linspace(0., 180., n, endpoint=False)
    s = radon(gt, theta=th, circle=CIRCLE)
    return s, iradon(s, theta=th, circle=CIRCLE)

# ---- Figure 1 ----
def figure1(n=32):
    s, r = radon_pair(G, n)
    fig, ax = plt.subplots(1, 3, figsize=(W_IN, 2.0))
    ax[0].imshow(G, cmap='gray', vmin=0, vmax=1); ax[0].set_title('(A) Ground-truth\nvessel image'); ax[0].axis('off')
    ax[1].imshow(s, cmap='viridis', aspect='auto'); ax[1].set_title(f'(B) Radon data\n({n} projection angles)')
    ax[1].set_xlabel('Projection angle index'); ax[1].set_ylabel('Detector position (pixels)')
    ax[2].imshow(r, cmap='gray', vmin=0, vmax=1); ax[2].set_title('(C) FBP reconstruction\n(streak artifacts)'); ax[2].axis('off')
    fig.tight_layout(); fig.savefig(f'{FIG}/fig1_sparse_view_problem.png', bbox_inches='tight'); plt.show()

# ---- Figure 2 ----
def figure2():
    fig, ax = plt.subplots(2, 4, figsize=(W_IN, 3.5), gridspec_kw=dict(hspace=1.0, wspace=0.7))
    ax[0, 0].imshow(G, cmap='gray', vmin=0, vmax=1); ax[0, 0].set_title('Ground truth')
    ax[1, 0].axis('off')
    ax[1, 0].text(0.0, 0.5, 'Top row:\nRadon transform\n\nBottom row:\nfiltered back-\nprojection (FBP)', fontsize=6.2,
                  va='center', ha='left', transform=ax[1, 0].transAxes, color='#1d2b3a')
    for j, n in enumerate(ANGLES):
        s, r = radon_pair(G, n)
        ax[0, j + 1].imshow(s, cmap='viridis', aspect='auto'); ax[0, j + 1].set_title(f'Radon data\n{n} angles')
        ax[0, j + 1].set_xlabel('Angle index')
        if j == 0: ax[0, j + 1].set_ylabel('Detector position (px)')
        else: ax[0, j + 1].set_yticks([])
        ax[1, j + 1].imshow(r, cmap='gray', vmin=0, vmax=1); ax[1, j + 1].set_title(f'FBP input\n{n} angles')
    for a in (ax[0, 0], ax[1, 1], ax[1, 2], ax[1, 3]): a.set_xticks([]); a.set_yticks([])
    fig.savefig(f'{FIG}/fig2_radon_fbp_workflow.png', bbox_inches='tight'); plt.show()

# ---- model outputs for one test image ----
def load_model(name, n, seed=SEEDS[0]):
    m = MODELS[name](in_ch=1, out_ch=1, base=32).to(device)
    m.load_state_dict(torch.load(f'{OUT}/ckpt/{PROTOCOL}_{name}_{n}_seed{seed}.pth', map_location=device)); return m.eval()

@torch.no_grad()
def panels(n):
    x = np.load(f'{OUT}/sparse{n}_test.npy', mmap_mode='r')[IDX].astype(np.float32)
    out = {'Sparse FBP\ninput': np.clip(x, 0, 1), 'Ground\ntruth': G}
    for name in MODELS:
        out[name if name != 'SimCBAM' else 'SimCBAM'] = load_model(name, n)(torch.from_numpy(x)[None, None].to(device)).clamp(0, 1).cpu().numpy()[0, 0]
    return out

# ---- Figure 5: 64 angles, two rows so every panel stays readable ----
def figure5(n=64):
    p = panels(n); keys = list(p)
    fig, ax = plt.subplots(2, 4, figsize=(W_IN, 2.85))
    for a in ax.ravel(): a.axis('off')
    for a, k in zip(ax.ravel(), keys):
        a.imshow(p[k], cmap='gray', vmin=0, vmax=1); a.set_title(k.replace('\n', ' '), fontsize=6.2)
    fig.tight_layout(); fig.savefig(f'{FIG}/fig5_qualitative_64.png', bbox_inches='tight'); plt.show()

# ---- Figure 6: all sampling levels (rows) x 7 columns ----
def figure6():
    fig, ax = plt.subplots(3, 7, figsize=(W_IN, 2.35))
    for r, n in enumerate(ANGLES):
        p = panels(n)
        for c, k in enumerate(p):
            ax[r, c].imshow(p[k], cmap='gray', vmin=0, vmax=1); ax[r, c].set_xticks([]); ax[r, c].set_yticks([])
            if r == 0: ax[r, c].set_title(k, fontsize=5.0)
        ax[r, 0].set_ylabel(f'{n} angles', fontsize=6)
    fig.tight_layout(pad=0.3); fig.savefig(f'{FIG}/fig6_qualitative_all.png', bbox_inches='tight'); plt.show()

# ---- Figure 7: PSNR / SSIM vs number of projection angles (error bars only if several seeds) ----
def figure7():
    fig, ax = plt.subplots(1, 2, figsize=(W_IN, 2.4))
    for m in MODELS:
        d = agg[agg.model == m].sort_values('angles')
        for a, (k, lab) in zip(ax, [('PSNR', 'PSNR (dB)'), ('SSIM', 'SSIM')]):
            sd = d[k + '_sd'].fillna(0).values
            a.errorbar(d.angles, d[k], yerr=sd if sd.any() else None, marker='o', ms=3, lw=1, capsize=2, label=m)
            a.set_xticks(ANGLES); a.set_xlabel('Number of projection angles'); a.set_ylabel(lab)
    ax[0].set_title('(a) PSNR'); ax[1].set_title('(b) SSIM'); ax[0].legend(fontsize=5.2, frameon=False)
    fig.tight_layout(); fig.savefig(f'{FIG}/fig7_metrics_vs_angles.png', bbox_inches='tight'); plt.show()

figure1(); figure2(); figure5(); figure6(); figure7()
print('figures saved to', FIG)

## 8. Old-bug demonstration figure (for the record)
Left: FBP input as it should be stored (float, clipped only for display). Right: what `astype(np.uint8)` produced in the original notebook.

In [ ]:
if os.path.exists(f'{ROOT}/test/0.tif'):
    g255 = load_gray(f'{ROOT}/test/0.tif') * 255
    th = np.linspace(0., 180., 32, endpoint=False)
    rec = iradon(radon(g255, theta=th, circle=True), theta=th, circle=True)
    fig, ax = plt.subplots(1, 2, figsize=(W_IN, 2.5))
    ax[0].imshow(np.clip(rec, 0, 255), cmap='gray', vmin=0, vmax=255); ax[0].set_title('float FBP (clipped for display)')
    ax[1].imshow(rec.astype(np.uint8), cmap='gray', vmin=0, vmax=255); ax[1].set_title('old code: astype(np.uint8)')
    for a in ax: a.axis('off')
    fig.tight_layout(); fig.savefig(f'{FIG}/old_uint8_bug_demo.png', bbox_inches='tight'); plt.show()
else:
    print('original .tif files not in this runtime: skipped (you already have this figure from your first run)')